# Intermediate Data Preparation and Processing

**Project:** Reliable supervised machine-learning models for lithium-ion battery State of Health (SOH) prediction

**Purpose of this notebook:** This notebook documents the preliminary data-processing workflow for the Intermediate Project Status Report. It uses `2_009.h5` as a pilot file from one battery cell. The workflow can later be applied to additional cell files for the final project.

> **Important scope note:** The results below are preliminary pilot results from cell `2_009`. They are not presented as results for the complete Zenodo dataset. The final project is planned to expand this workflow to approximately 15 cells and evaluate generalization to unseen cells.

In [ ]:
# Install/import packages if needed
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
DATA_PATH = Path("2_009.h5")
if not DATA_PATH.exists():
    candidates = list(Path(".").glob("**/2_009.h5"))
    if candidates:
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError("Place 2_009.h5 in the notebook folder or update DATA_PATH.")

print(f"Using: {DATA_PATH.resolve()}")
print(f"File size: {DATA_PATH.stat().st_size / 1024**2:.1f} MB")

## 1. Inspect the HDF5 structure

The file contains cell-level metadata in `info` and measurement arrays in the `tests` group. We inspect the structure before processing so that the extraction decisions are transparent and reproducible.

In [ ]:
with h5py.File(DATA_PATH, "r") as f:
    print("Root objects:", list(f.keys()))
    print("Number of test groups:", len(f["tests"]))
    print("Info shape:", f["info"].shape)
    print("Info fields:", f["info"].dtype.names)
    first_test = sorted(f["tests"].keys())[0]
    print(f"\nExample group: {first_test}")
    print("Datasets:", list(f["tests"][first_test].keys()))
    for name, ds in f["tests"][first_test].items():
        print(f"  {name}: shape={ds.shape}, dtype={ds.dtype}")

## 2. Build a test metadata table

For this pilot, each HDF5 test group is treated as one test record. The metadata table preserves the cell ID, test ID, experiment/profile name, and schedule information.

In [ ]:
def decode(value):
    return value.decode("utf-8") if isinstance(value, (bytes, np.bytes_)) else value

with h5py.File(DATA_PATH, "r") as f:
    info = f["info"][:]
    test_names = sorted(f["tests"].keys())

metadata_rows = []
for row, test_name in zip(info, test_names):
    metadata_rows.append({
        "test_group": test_name,
        "cell_id": decode(row["idcell"]),
        "idtest": int(row["idtest"]),
        "idexp": int(row["idexp"]),
        "experiment": decode(row["expname"]),
        "start_datetime": float(row["startdatetime"]),
        "schedule_file": decode(row["schedulefilename"]),
    })
metadata_df = pd.DataFrame(metadata_rows)
metadata_df.head()

In [ ]:
print("Unique cells:", metadata_df["cell_id"].unique().tolist())
print("Number of tests:", len(metadata_df))
print("Experiments/profiles:")
display(metadata_df["experiment"].value_counts().sort_index().to_frame("number_of_tests"))

## 3. Data-quality checks

The raw arrays are large, so the checks are performed test by test. We check for missing/non-finite values, invalid temperature sentinel values near `-273.16`, and non-increasing time values. Time irregularities are flagged rather than silently deleting measurements.

In [ ]:
quality_rows = []
summary = {"total_rows": 0, "nan_inf_values": 0, "temperature_sentinel_values": 0, "non_increasing_time_transitions": 0}

with h5py.File(DATA_PATH, "r") as f:
    for meta in metadata_rows:
        test_name = meta["test_group"]
        g = f["tests"][test_name]
        arrays = {name: g[name][:] for name in ["voltage", "current", "temperature", "test_time", "chargecap", "dischargecap", "stepindex"]}
        n = len(arrays["voltage"])
        nonfinite = sum(int((~np.isfinite(values)).sum()) for values in arrays.values() if np.issubdtype(values.dtype, np.number))
        sentinel = int(np.isclose(arrays["temperature"], -273.16, atol=0.01).sum())
        time_bad = int((np.diff(arrays["test_time"]) <= 0).sum()) if n > 1 else 0
        quality_rows.append({**meta, "rows": n, "nonfinite_values": nonfinite, "temperature_sentinel_values": sentinel, "non_increasing_time_transitions": time_bad})
        summary["total_rows"] += n
        summary["nan_inf_values"] += nonfinite
        summary["temperature_sentinel_values"] += sentinel
        summary["non_increasing_time_transitions"] += time_bad

quality_df = pd.DataFrame(quality_rows)
print(summary)
print("Tests with non-finite values:", int((quality_df.nonfinite_values > 0).sum()))
print("Tests with temperature sentinels:", int((quality_df.temperature_sentinel_values > 0).sum()))
print("Tests with non-increasing time:", int((quality_df.non_increasing_time_transitions > 0).sum()))
display(quality_df[quality_df[["temperature_sentinel_values", "non_increasing_time_transitions"]].sum(axis=1) > 0][["test_group", "experiment", "temperature_sentinel_values", "non_increasing_time_transitions"]])

### Cleaning decisions for the pilot

* Non-finite values are counted and would be removed from calculations if present.
* Temperature values near `-273.16` are treated as invalid sentinel values and excluded from temperature summaries.
* Non-increasing time transitions are flagged for review. They are not automatically used as a reason to delete an entire test because they can occur at measurement-step transitions.
* The original identifiers and test groups are retained so later analyses can audit the processing.

## 4. Create a compact test-level feature table

Because the raw file contains millions of measurement rows, this pilot creates one row per test. The features summarize voltage, current, temperature, and capacity. The maximum valid discharge capacity is used as a preliminary capacity estimate. This is a transparent pilot representation; the final workflow can use a more detailed cycle-level table after the group confirms the exact cycle definition.

In [ ]:
feature_rows = []
with h5py.File(DATA_PATH, "r") as f:
    for meta in metadata_rows:
        g = f["tests"][meta["test_group"]]
        voltage = g["voltage"][:].astype(float)
        current = g["current"][:].astype(float)
        temperature = g["temperature"][:].astype(float)
        test_time = g["test_time"][:].astype(float)
        chargecap = g["chargecap"][:].astype(float)
        dischargecap = g["dischargecap"][:].astype(float)
        valid_temp = temperature[np.isfinite(temperature) & ~np.isclose(temperature, -273.16, atol=0.01)]
        valid_discharge = dischargecap[np.isfinite(dischargecap) & (dischargecap > 0)]
        valid_charge = chargecap[np.isfinite(chargecap) & (chargecap > 0)]
        valid_voltage = voltage[np.isfinite(voltage)]
        valid_current = current[np.isfinite(current)]
        feature_rows.append({
            **meta,
            "n_measurements": len(voltage),
            "duration_seconds": float(np.nanmax(test_time) - np.nanmin(test_time)),
            "voltage_mean": float(np.mean(valid_voltage)) if len(valid_voltage) else np.nan,
            "voltage_min": float(np.min(valid_voltage)) if len(valid_voltage) else np.nan,
            "voltage_max": float(np.max(valid_voltage)) if len(valid_voltage) else np.nan,
            "current_mean": float(np.mean(valid_current)) if len(valid_current) else np.nan,
            "current_min": float(np.min(valid_current)) if len(valid_current) else np.nan,
            "current_max": float(np.max(valid_current)) if len(valid_current) else np.nan,
            "temperature_mean_clean": float(np.mean(valid_temp)) if len(valid_temp) else np.nan,
            "temperature_min_clean": float(np.min(valid_temp)) if len(valid_temp) else np.nan,
            "temperature_max_clean": float(np.max(valid_temp)) if len(valid_temp) else np.nan,
            "charge_capacity_max": float(np.max(valid_charge)) if len(valid_charge) else np.nan,
            "discharge_capacity_max": float(np.max(valid_discharge)) if len(valid_discharge) else np.nan,
        })

features_df = pd.DataFrame(feature_rows)
features_df["capacity_estimate"] = features_df["discharge_capacity_max"].fillna(features_df["charge_capacity_max"])
first_capacity = features_df["capacity_estimate"].dropna().iloc[0]
features_df["soh_proxy_percent"] = 100 * features_df["capacity_estimate"] / first_capacity
features_df[["test_group", "experiment", "n_measurements", "capacity_estimate", "soh_proxy_percent"]].head()

In [ ]:
print("Feature table shape:", features_df.shape)
print("SOH proxy range:", round(features_df["soh_proxy_percent"].min(), 2), "to", round(features_df["soh_proxy_percent"].max(), 2))
print("Rows with missing capacity estimate:", int(features_df["capacity_estimate"].isna().sum()))
display(features_df.describe(include="all").T.head(20))

In [ ]:
# Preliminary visualization of the capacity/SOH proxy by test order
plot_df = features_df.reset_index(drop=True).copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(plot_df.index, plot_df["capacity_estimate"], marker=".", linewidth=1)
axes[0].set_title("Pilot capacity estimate by test order")
axes[0].set_xlabel("Test order")
axes[0].set_ylabel("Capacity estimate")
axes[1].plot(plot_df.index, plot_df["soh_proxy_percent"], marker=".", linewidth=1)
axes[1].axhline(80, color="red", linestyle="--", label="80% reference")
axes[1].set_title("Preliminary SOH proxy by test order")
axes[1].set_xlabel("Test order")
axes[1].set_ylabel("SOH proxy (%)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Export files for the group

The exported CSV is a compact, shareable pilot table. It is suitable for beginning preliminary regression/classification experiments. Before final modeling, the group should confirm the final target definition, split design, and whether the final table will be test-level or cycle-level.

In [ ]:
OUTPUT_PATH = Path("pilot_features_2_009.csv")
features_df.to_csv(OUTPUT_PATH, index=False)
quality_df.to_csv("pilot_quality_by_test_2_009.csv", index=False)
print(f"Saved {OUTPUT_PATH.resolve()}")
print(f"Saved {Path('pilot_quality_by_test_2_009.csv').resolve()}")

## 6. Progress summary for the Intermediate Status Report

This pilot completed the initial HDF5 inspection, metadata extraction, data-quality checks, invalid-temperature flagging, compact feature creation, and a preliminary SOH proxy calculation for one cell. The raw file contained approximately 13.7 million measurement rows across 57 tests. No non-finite values were found in the inspected arrays; 51 temperature readings were identified as the `-273.16` sentinel, and four non-increasing time transitions were flagged for review.

The next step is to apply this documented workflow to additional cell files. The final project will use approximately 15 cells and will evaluate models on held-out cells and operating profiles.